# ViT SMALL Experiments - Background Reliance
- Runs the ViT architecture, trained with and without background randomisation agumentation across multiple seeds

# Importing Libraries

In [1]:
import pathlib
import json
import torch
import numpy as np
import random
import pandas as pd

from cory_data_wrapper import *
from cory_utils import *
import tensorboard
import time

# Select Device

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device}")

Using cuda


# Create constants and create paths

In [3]:
# Make sure that the directory exists
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
SEEDS = [0, 339, 1, 2, 3, 4, 5 ,6]
VIT_ARCHITECTURE = ["vit_small_patch16_224.augreg_in1k"]

DATA_DIR = pathlib.Path("data")
DATA_ROUTE = DATA_DIR / "rebuild" / "in9_train"
VAL_ROUTE = DATA_DIR / "rebuild" /"in9_val"
BOUNDING_BOX_PATH = DATA_DIR / "LOC_train_solution.csv"
TEST_INSTANCES = DATA_DIR / "bg_challenge_clean"
RESULTS = pathlib.Path("results_10_epochs")
TENSORBOARD_DIR = pathlib.Path("runs_10_epochs")
FIGURES = pathlib.Path("figures_10_epochs")
RESULTS.mkdir(exist_ok=True)
TENSORBOARD_DIR.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

In [4]:
print(ViTModel("vit_small_patch16_224.augreg_in1k", num_classes=9, pretrained=False).backbone.pretrained_cfg["mean"])
m = ViTModel("vit_small_patch16_224.augreg_in1k", num_classes=9, pretrained=True)
cfg = m.backbone.pretrained_cfg
print(cfg["architecture"], cfg["tag"])

C:\Users\minio\PycharmProjects\JupyterProject2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(0.5, 0.5, 0.5)


vit_small_patch16_224 augreg_in1k


# Load the data

In [4]:
# Load data
bboxs = load_bbox_csv(BOUNDING_BOX_PATH)
base_train_dataset = IN9BoundingBoxDataset(DATA_ROUTE, bboxs, image_size=IMAGE_SIZE)
train_class_to_idx = base_train_dataset.class_to_idx

# print the instances and num of classes
print(f"Loaded {len(base_train_dataset)} images", "with" , len(base_train_dataset.classes), "classes")

Loaded 72036 images with 9 classes


# Split the data (Train, Validation)

In [5]:
train_subset = base_train_dataset

# Create data loaders for all sets, for faster loading and shuffling and batches

# Create the validation loader
val_dataset_normalised = ImageFolderEval(VAL_ROUTE, image_size=IMAGE_SIZE, mean = VIT_MEAN, std = VIT_STD)
val_loader = make_loader(val_dataset_normalised, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"Loaded {len(val_dataset_normalised)} validation images")
assert val_dataset_normalised.class_to_idx == train_class_to_idx
print(val_dataset_normalised.class_to_idx)
# make loaders for the test set, one for each variant
test_variant_loaders = {
    variant: make_loader(
        fix_test_labels(
            ImageFolderEval(TEST_INSTANCES / variant / "val", image_size=IMAGE_SIZE, mean= VIT_MEAN, std= VIT_STD),train_class_to_idx
        ),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=0
    )
    for variant in ["original", "no_fg", "mixed_same", "mixed_rand"]
}
print("Test variants loaders created:", list(test_variant_loaders.keys()))
# Print out the test variant loader size
for variant, loader in test_variant_loaders.items():
    print(f"{variant}: {len(loader.dataset)} images")

Loaded 1886 validation images
{'Bird': 0, 'Carnivore': 1, 'Dog': 2, 'Fish': 3, 'Insect': 4, 'Instrument': 5, 'Primate': 6, 'Reptile': 7, 'Vehicle': 8}
Test variants loaders created: ['original', 'no_fg', 'mixed_same', 'mixed_rand']
original: 3889 images
no_fg: 3889 images
mixed_same: 3889 images
mixed_rand: 3889 images


# Perform Learning Rate Search

- Train on the original images for up to 10 epochs (early stopping, patience 3) with each of 3 learning rates (2.5e-5, 5e-5, 1e-4), seed 0.
- The best learning rate is written into BEST_LR so the search is skipped on later runs.

In [7]:
# Create the dictionary for best Learning-Rate
BEST_LR = {
    "vit_small_patch16_224.augreg_in1k": 2.5e-05
}
best_lr_per_arch = {
    "vit_small_patch16_224.augreg_in1k": 2.5e-05
}
# loop through all architectures and perform the Hyper Parameter search for each architecture
for arch_name in VIT_ARCHITECTURE:
    # Only run the search if the learning rate is not already known
    if BEST_LR[arch_name] is not None:
        best_lr_per_arch[arch_name] = BEST_LR[arch_name]
        continue

    print(f"\n{'-'*60}\n Hyper Parameter Search: {arch_name} \n{'-'*60}")
    #
    search_train_dataset = BackgroundRandomizationDataset(train_subset, augmentation_strength=0.0, seed=0, mean = VIT_MEAN, std = VIT_STD)
    search_train_loader = make_loader(search_train_dataset,batch_size=BATCH_SIZE, shuffle=True, num_workers=12, persistent_workers=True)

    best_lr, hyperparameter_trials = hyperparameter_search(
        lambda a = arch_name: ViTModel(a, num_classes=9, pretrained=True),
        search_train_loader, val_loader, device,
        learning_rates=(2.5e-5, 5e-5, 1e-4), search_epochs=10,
        log_dir=TENSORBOARD_DIR / "hpsearch" / arch_name
    )
    best_lr_per_arch[arch_name] = best_lr
    print(f"{arch_name} best LR: {best_lr:}")

print("\nBest LR per architecture:", best_lr_per_arch)


------------------------------------------------------------
 Hyper Parameter Search: vit_small_patch16_224.augreg_in1k 
------------------------------------------------------------

--- HP search: lr=2.5e-05 ---
[hpsearch_lr2.5e-05] Epoch 1/10: train_loss=0.4315  train_acc=0.8654 val_loss=0.1095  val_acc=0.9740
[hpsearch_lr2.5e-05] Epoch 2/10: train_loss=0.0437  train_acc=0.9888 val_loss=0.1166  val_acc=0.9687
[hpsearch_lr2.5e-05] Epoch 3/10: train_loss=0.0189  train_acc=0.9946 val_loss=0.1307  val_acc=0.9698
[hpsearch_lr2.5e-05] Epoch 4/10: train_loss=0.0092  train_acc=0.9973 val_loss=0.1794  val_acc=0.9698
[hpsearch_lr2.5e-05] Early stopping at epoch 4

--- HP search: lr=5e-05 ---
[hpsearch_lr5e-05] Epoch 1/10: train_loss=0.3429  train_acc=0.8942 val_loss=0.1255  val_acc=0.9661
[hpsearch_lr5e-05] Epoch 2/10: train_loss=0.0758  train_acc=0.9788 val_loss=0.1920  val_acc=0.9480
[hpsearch_lr5e-05] Epoch 3/10: train_loss=0.0483  train_acc=0.9869 val_loss=0.1677  val_acc=0.9613
[hpsearch

In [8]:
best_lr_per_arch = {
    "vit_small_patch16_224.augreg_in1k": 2.5e-05
}

In [9]:
# Architecture Train, Eval and Test Function

def run_vit_experiment(arch_name ="vit_small_patch16_224.augreg_in1k", augmented = False, seed= 0):
    """
    Run the ViT experiment for a specified architecture and seed.
    Stores the data in results_10_epochs

    @param arch_name: Name of the ViT architecture.
    @param augmented: Whether to use augmented or original data.
    @param seed: Seed for reproducibility.
    @return: A dictionary containing the results of the specified experiment
    """
    data_type = "augmented" if augmented else "original"
    run_name = f"{arch_name}_{data_type}_{seed}"
    print(f"\n{'='*60}\n{run_name}\n{'='*60}")

    # If data is augmented, strength = 1 (swap) else 0 (do not swap background), swap the background with a random image's background
    train_dataset_for_run = BackgroundRandomizationDataset(
        train_subset, augmentation_strength=1.0 if augmented else 0.0, seed=seed, mean = VIT_MEAN, std = VIT_STD
    )
    train_loader = make_loader(train_dataset_for_run, batch_size=BATCH_SIZE, shuffle=True, num_workers=12, persistent_workers=True)

    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    # Define the model and move it to the device
    model = ViTModel(arch_name, num_classes=9, pretrained=True).to(device)

    start_time = time.time()
    # Train the model and store the results
    trained_model, history = train_model(
        model, train_loader, val_loader, device, epochs=EPOCHS, early_stopping_patience=3, model_name=run_name, lr=best_lr_per_arch[arch_name], log_dir=TENSORBOARD_DIR / run_name
    )
    training_time_in_secs = time.time() - start_time

    # Save the model weights
    weights_path = RESULTS / f"{run_name}_weights.pt"
    torch.save(trained_model.state_dict(), weights_path)

    variant_accuracies = evaluate_on_variants(trained_model, test_variant_loaders, device)
    # compute the reliance metrics
    reliance = compute_reliance_metrics(variant_accuracies, num_classes=9)
    print(f"Test variants: {variant_accuracies}")
    print(f"BG-Gap: {reliance['bg_gap']:.4f}, No-FG: {reliance['no_fg_accuracy']:.4f}, Original: {reliance['original_accuracy']:.4f}")
    # Store the results in a dictionary
    results = {
        "architecture" : arch_name,
        "data_type" : data_type,
        "seed" : seed,
        "variant_accuracies" : variant_accuracies,
        "reliance_metrics" : reliance,
        "final_val_accuracy" : history["val_accuracy"][-1],
        "epochs_run" : len(history["val_accuracy"]),
        "training_time_in_secs" : training_time_in_secs,
        "best_val_accuracy" : max(history["val_accuracy"]),
    }
    # Add this run to the results json, replacing any earlier run with the same name
    results_path = RESULTS / "vit_small_results.json"
    results_each_run = {}
    if results_path.exists():
        with open(results_path, "r") as f:
            for r in json.load(f):
                key = f"{r['architecture']}_{r['data_type']}_{r['seed']}"
                results_each_run[key] = r
    results_each_run[run_name] = results
    with open(results_path, "w") as f:
        json.dump(list(results_each_run.values()), f, indent=2)

    return results

# Run each seed and condition
Each cell trains one model for one seed and one condition, saves the weights to results_10_epochs/, evaluates it on the four test variants and adds the result to the results json. Ten seeds per condition: 0, 339, 1–8.

# ViT Small (No Augmentation)

In [10]:
run_vit_experiment(arch_name= "vit_small_patch16_224.augreg_in1k",augmented=False, seed=0)


vit_small_patch16_224.augreg_in1k_original_0
[vit_small_patch16_224.augreg_in1k_original_0] Epoch 1/10: train_loss=0.4316  train_acc=0.8658 val_loss=0.1046  val_acc=0.9719
[vit_small_patch16_224.augreg_in1k_original_0] Epoch 2/10: train_loss=0.0442  train_acc=0.9881 val_loss=0.1307  val_acc=0.9666
[vit_small_patch16_224.augreg_in1k_original_0] Epoch 3/10: train_loss=0.0186  train_acc=0.9950 val_loss=0.1380  val_acc=0.9714
[vit_small_patch16_224.augreg_in1k_original_0] Epoch 4/10: train_loss=0.0099  train_acc=0.9973 val_loss=0.1771  val_acc=0.9682
[vit_small_patch16_224.augreg_in1k_original_0] Early stopping at epoch 4
Test variants: {'original': 0.9866289534584726, 'no_fg': 0.7361789663152482, 'mixed_same': 0.9506299820005143, 'mixed_rand': 0.870917973772178}
BG-Gap: 0.0797, No-FG: 0.7362, Original: 0.9866


{'architecture': 'vit_small_patch16_224.augreg_in1k',
 'data_type': 'original',
 'seed': 0,
 'variant_accuracies': {'original': 0.9866289534584726,
  'no_fg': 0.7361789663152482,
  'mixed_same': 0.9506299820005143,
  'mixed_rand': 0.870917973772178},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.7361789663152482,
  'bg_gap': 0.07971200822833635,
  'original_accuracy': 0.9866289534584726},
 'final_val_accuracy': 0.968186638388123,
 'epochs_run': 4,
 'training_time_in_secs': 435.77464747428894,
 'best_val_accuracy': 0.971898197242842}

In [11]:
run_vit_experiment(arch_name= "vit_small_patch16_224.augreg_in1k",augmented=False, seed=339)


vit_small_patch16_224.augreg_in1k_original_339
[vit_small_patch16_224.augreg_in1k_original_339] Epoch 1/10: train_loss=0.4409  train_acc=0.8600 val_loss=0.1014  val_acc=0.9703
[vit_small_patch16_224.augreg_in1k_original_339] Epoch 2/10: train_loss=0.0433  train_acc=0.9882 val_loss=0.1266  val_acc=0.9682
[vit_small_patch16_224.augreg_in1k_original_339] Epoch 3/10: train_loss=0.0186  train_acc=0.9947 val_loss=0.1417  val_acc=0.9730
[vit_small_patch16_224.augreg_in1k_original_339] Epoch 4/10: train_loss=0.0095  train_acc=0.9973 val_loss=0.1684  val_acc=0.9714
[vit_small_patch16_224.augreg_in1k_original_339] Epoch 5/10: train_loss=0.0060  train_acc=0.9984 val_loss=0.1468  val_acc=0.9735
[vit_small_patch16_224.augreg_in1k_original_339] Epoch 6/10: train_loss=0.0030  train_acc=0.9993 val_loss=0.1751  val_acc=0.9756
[vit_small_patch16_224.augreg_in1k_original_339] Epoch 7/10: train_loss=0.0007  train_acc=0.9998 val_loss=0.1793  val_acc=0.9708
[vit_small_patch16_224.augreg_in1k_original_339] 

{'architecture': 'vit_small_patch16_224.augreg_in1k',
 'data_type': 'original',
 'seed': 339,
 'variant_accuracies': {'original': 0.9861146824376447,
  'no_fg': 0.6996657238364619,
  'mixed_same': 0.9508871175109282,
  'mixed_rand': 0.8729750578554898},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.6996657238364619,
  'bg_gap': 0.07791205965543846,
  'original_accuracy': 0.9861146824376447},
 'final_val_accuracy': 0.9777306468716861,
 'epochs_run': 10,
 'training_time_in_secs': 1038.17755818367,
 'best_val_accuracy': 0.9793213149522799}

In [12]:
run_vit_experiment(arch_name= "vit_small_patch16_224.augreg_in1k",augmented=False, seed=1)


vit_small_patch16_224.augreg_in1k_original_1
[vit_small_patch16_224.augreg_in1k_original_1] Epoch 1/10: train_loss=0.4132  train_acc=0.8678 val_loss=0.1036  val_acc=0.9719
[vit_small_patch16_224.augreg_in1k_original_1] Epoch 2/10: train_loss=0.0444  train_acc=0.9879 val_loss=0.1224  val_acc=0.9671
[vit_small_patch16_224.augreg_in1k_original_1] Epoch 3/10: train_loss=0.0172  train_acc=0.9952 val_loss=0.1373  val_acc=0.9724
[vit_small_patch16_224.augreg_in1k_original_1] Epoch 4/10: train_loss=0.0109  train_acc=0.9970 val_loss=0.1646  val_acc=0.9719
[vit_small_patch16_224.augreg_in1k_original_1] Epoch 5/10: train_loss=0.0057  train_acc=0.9984 val_loss=0.1792  val_acc=0.9714
[vit_small_patch16_224.augreg_in1k_original_1] Epoch 6/10: train_loss=0.0021  train_acc=0.9995 val_loss=0.1722  val_acc=0.9740
[vit_small_patch16_224.augreg_in1k_original_1] Epoch 7/10: train_loss=0.0010  train_acc=0.9997 val_loss=0.1644  val_acc=0.9751
[vit_small_patch16_224.augreg_in1k_original_1] Epoch 8/10: train_

{'architecture': 'vit_small_patch16_224.augreg_in1k',
 'data_type': 'original',
 'seed': 1,
 'variant_accuracies': {'original': 0.9881717665209565,
  'no_fg': 0.7287220365132425,
  'mixed_same': 0.9439444587297505,
  'mixed_rand': 0.8737464643867318},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.7287220365132425,
  'bg_gap': 0.07019799434301877,
  'original_accuracy': 0.9881717665209565},
 'final_val_accuracy': 0.9761399787910923,
 'epochs_run': 10,
 'training_time_in_secs': 1046.3126707077026,
 'best_val_accuracy': 0.9761399787910923}

In [13]:
run_vit_experiment(arch_name= "vit_small_patch16_224.augreg_in1k",augmented=False, seed=2)



vit_small_patch16_224.augreg_in1k_original_2


C:\Users\minio\PycharmProjects\JupyterProject2\.venv\Lib\site-packages\torch\optim\lr_scheduler.py:1194: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


[vit_small_patch16_224.augreg_in1k_original_2] Epoch 1/10: train_loss=0.4357  train_acc=0.8619 val_loss=0.0945  val_acc=0.9767
[vit_small_patch16_224.augreg_in1k_original_2] Epoch 2/10: train_loss=0.0441  train_acc=0.9878 val_loss=0.1307  val_acc=0.9692
[vit_small_patch16_224.augreg_in1k_original_2] Epoch 3/10: train_loss=0.0172  train_acc=0.9951 val_loss=0.1522  val_acc=0.9698
[vit_small_patch16_224.augreg_in1k_original_2] Epoch 4/10: train_loss=0.0107  train_acc=0.9971 val_loss=0.1434  val_acc=0.9751
[vit_small_patch16_224.augreg_in1k_original_2] Early stopping at epoch 4
Test variants: {'original': 0.9868860889688866, 'no_fg': 0.7258935458986886, 'mixed_same': 0.9521727950629982, 'mixed_rand': 0.8763178194908717}
BG-Gap: 0.0759, No-FG: 0.7259, Original: 0.9869


{'architecture': 'vit_small_patch16_224.augreg_in1k',
 'data_type': 'original',
 'seed': 2,
 'variant_accuracies': {'original': 0.9868860889688866,
  'no_fg': 0.7258935458986886,
  'mixed_same': 0.9521727950629982,
  'mixed_rand': 0.8763178194908717},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.7258935458986886,
  'bg_gap': 0.0758549755721265,
  'original_accuracy': 0.9868860889688866},
 'final_val_accuracy': 0.9750795334040296,
 'epochs_run': 4,
 'training_time_in_secs': 433.3434751033783,
 'best_val_accuracy': 0.9766702014846236}

In [14]:
run_vit_experiment(arch_name= "vit_small_patch16_224.augreg_in1k",augmented=False, seed=3)



vit_small_patch16_224.augreg_in1k_original_3
[vit_small_patch16_224.augreg_in1k_original_3] Epoch 1/10: train_loss=0.4439  train_acc=0.8574 val_loss=0.1047  val_acc=0.9724
[vit_small_patch16_224.augreg_in1k_original_3] Epoch 2/10: train_loss=0.0453  train_acc=0.9880 val_loss=0.1177  val_acc=0.9730
[vit_small_patch16_224.augreg_in1k_original_3] Epoch 3/10: train_loss=0.0190  train_acc=0.9951 val_loss=0.1130  val_acc=0.9767
[vit_small_patch16_224.augreg_in1k_original_3] Epoch 4/10: train_loss=0.0108  train_acc=0.9970 val_loss=0.1555  val_acc=0.9719
[vit_small_patch16_224.augreg_in1k_original_3] Epoch 5/10: train_loss=0.0061  train_acc=0.9984 val_loss=0.1578  val_acc=0.9756
[vit_small_patch16_224.augreg_in1k_original_3] Epoch 6/10: train_loss=0.0030  train_acc=0.9991 val_loss=0.1477  val_acc=0.9761
[vit_small_patch16_224.augreg_in1k_original_3] Early stopping at epoch 6
Test variants: {'original': 0.9884289020313706, 'no_fg': 0.7410645410131139, 'mixed_same': 0.9390588840318848, 'mixed_r

{'architecture': 'vit_small_patch16_224.augreg_in1k',
 'data_type': 'original',
 'seed': 3,
 'variant_accuracies': {'original': 0.9884289020313706,
  'no_fg': 0.7410645410131139,
  'mixed_same': 0.9390588840318848,
  'mixed_rand': 0.8647467215222422},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.7410645410131139,
  'bg_gap': 0.07431216250964257,
  'original_accuracy': 0.9884289020313706},
 'final_val_accuracy': 0.9761399787910923,
 'epochs_run': 6,
 'training_time_in_secs': 635.0663850307465,
 'best_val_accuracy': 0.9766702014846236}

In [15]:
run_vit_experiment(arch_name= "vit_small_patch16_224.augreg_in1k",augmented=False, seed=4)


vit_small_patch16_224.augreg_in1k_original_4
[vit_small_patch16_224.augreg_in1k_original_4] Epoch 1/10: train_loss=0.4580  train_acc=0.8514 val_loss=0.1130  val_acc=0.9692
[vit_small_patch16_224.augreg_in1k_original_4] Epoch 2/10: train_loss=0.0436  train_acc=0.9881 val_loss=0.1354  val_acc=0.9719
[vit_small_patch16_224.augreg_in1k_original_4] Epoch 3/10: train_loss=0.0179  train_acc=0.9950 val_loss=0.1176  val_acc=0.9756
[vit_small_patch16_224.augreg_in1k_original_4] Epoch 4/10: train_loss=0.0085  train_acc=0.9977 val_loss=0.1599  val_acc=0.9692
[vit_small_patch16_224.augreg_in1k_original_4] Epoch 5/10: train_loss=0.0057  train_acc=0.9985 val_loss=0.1681  val_acc=0.9745
[vit_small_patch16_224.augreg_in1k_original_4] Epoch 6/10: train_loss=0.0029  train_acc=0.9992 val_loss=0.1453  val_acc=0.9761
[vit_small_patch16_224.augreg_in1k_original_4] Epoch 7/10: train_loss=0.0008  train_acc=0.9997 val_loss=0.1642  val_acc=0.9761
[vit_small_patch16_224.augreg_in1k_original_4] Epoch 8/10: train_

{'architecture': 'vit_small_patch16_224.augreg_in1k',
 'data_type': 'original',
 'seed': 4,
 'variant_accuracies': {'original': 0.9894574440730265,
  'no_fg': 0.7269220879403445,
  'mixed_same': 0.9465158138338905,
  'mixed_rand': 0.8601182823347905},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.7269220879403445,
  'bg_gap': 0.08639753149910001,
  'original_accuracy': 0.9894574440730265},
 'final_val_accuracy': 0.9798515376458112,
 'epochs_run': 10,
 'training_time_in_secs': 1040.8327629566193,
 'best_val_accuracy': 0.9798515376458112}

In [16]:
run_vit_experiment(arch_name="vit_small_patch16_224.augreg_in1k", augmented=False, seed=5)


vit_small_patch16_224.augreg_in1k_original_5
[vit_small_patch16_224.augreg_in1k_original_5] Epoch 1/10: train_loss=0.4450  train_acc=0.8600 val_loss=0.1026  val_acc=0.9730
[vit_small_patch16_224.augreg_in1k_original_5] Epoch 2/10: train_loss=0.0436  train_acc=0.9886 val_loss=0.1277  val_acc=0.9692
[vit_small_patch16_224.augreg_in1k_original_5] Epoch 3/10: train_loss=0.0187  train_acc=0.9951 val_loss=0.1239  val_acc=0.9761
[vit_small_patch16_224.augreg_in1k_original_5] Epoch 4/10: train_loss=0.0106  train_acc=0.9971 val_loss=0.1430  val_acc=0.9698
[vit_small_patch16_224.augreg_in1k_original_5] Epoch 5/10: train_loss=0.0057  train_acc=0.9983 val_loss=0.1645  val_acc=0.9724
[vit_small_patch16_224.augreg_in1k_original_5] Epoch 6/10: train_loss=0.0023  train_acc=0.9994 val_loss=0.1632  val_acc=0.9735
[vit_small_patch16_224.augreg_in1k_original_5] Early stopping at epoch 6
Test variants: {'original': 0.9814862432501928, 'no_fg': 0.6988943173052199, 'mixed_same': 0.9372589354589869, 'mixed_r

{'architecture': 'vit_small_patch16_224.augreg_in1k',
 'data_type': 'original',
 'seed': 5,
 'variant_accuracies': {'original': 0.9814862432501928,
  'no_fg': 0.6988943173052199,
  'mixed_same': 0.9372589354589869,
  'mixed_rand': 0.8624325019285163},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.6988943173052199,
  'bg_gap': 0.07482643353047058,
  'original_accuracy': 0.9814862432501928},
 'final_val_accuracy': 0.9734888653234358,
 'epochs_run': 6,
 'training_time_in_secs': 639.5551924705505,
 'best_val_accuracy': 0.9761399787910923}

In [17]:
run_vit_experiment(arch_name="vit_small_patch16_224.augreg_in1k", augmented=False, seed=6)


vit_small_patch16_224.augreg_in1k_original_6
[vit_small_patch16_224.augreg_in1k_original_6] Epoch 1/10: train_loss=0.4311  train_acc=0.8614 val_loss=0.1083  val_acc=0.9671
[vit_small_patch16_224.augreg_in1k_original_6] Epoch 2/10: train_loss=0.0426  train_acc=0.9882 val_loss=0.1402  val_acc=0.9661
[vit_small_patch16_224.augreg_in1k_original_6] Epoch 3/10: train_loss=0.0193  train_acc=0.9948 val_loss=0.1335  val_acc=0.9703
[vit_small_patch16_224.augreg_in1k_original_6] Epoch 4/10: train_loss=0.0098  train_acc=0.9974 val_loss=0.1497  val_acc=0.9730
[vit_small_patch16_224.augreg_in1k_original_6] Epoch 5/10: train_loss=0.0052  train_acc=0.9985 val_loss=0.1487  val_acc=0.9756
[vit_small_patch16_224.augreg_in1k_original_6] Epoch 6/10: train_loss=0.0024  train_acc=0.9993 val_loss=0.1704  val_acc=0.9724
[vit_small_patch16_224.augreg_in1k_original_6] Epoch 7/10: train_loss=0.0011  train_acc=0.9998 val_loss=0.1748  val_acc=0.9756
[vit_small_patch16_224.augreg_in1k_original_6] Epoch 8/10: train_

{'architecture': 'vit_small_patch16_224.augreg_in1k',
 'data_type': 'original',
 'seed': 6,
 'variant_accuracies': {'original': 0.9863718179480586,
  'no_fg': 0.7053227050655696,
  'mixed_same': 0.9475443558755464,
  'mixed_rand': 0.870146567240936},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.7053227050655696,
  'bg_gap': 0.07739778863461044,
  'original_accuracy': 0.9863718179480586},
 'final_val_accuracy': 0.9745493107104984,
 'epochs_run': 8,
 'training_time_in_secs': 839.0707836151123,
 'best_val_accuracy': 0.975609756097561}

In [18]:
run_vit_experiment(arch_name="vit_small_patch16_224.augreg_in1k", augmented=False, seed=7)


vit_small_patch16_224.augreg_in1k_original_7
[vit_small_patch16_224.augreg_in1k_original_7] Epoch 1/10: train_loss=0.4231  train_acc=0.8679 val_loss=0.1010  val_acc=0.9735
[vit_small_patch16_224.augreg_in1k_original_7] Epoch 2/10: train_loss=0.0448  train_acc=0.9880 val_loss=0.1284  val_acc=0.9708
[vit_small_patch16_224.augreg_in1k_original_7] Epoch 3/10: train_loss=0.0175  train_acc=0.9955 val_loss=0.1713  val_acc=0.9677
[vit_small_patch16_224.augreg_in1k_original_7] Epoch 4/10: train_loss=0.0112  train_acc=0.9971 val_loss=0.1504  val_acc=0.9703
[vit_small_patch16_224.augreg_in1k_original_7] Early stopping at epoch 4
Test variants: {'original': 0.9845718693751607, 'no_fg': 0.7487786063255335, 'mixed_same': 0.9480586268963744, 'mixed_rand': 0.8621753664181023}
BG-Gap: 0.0859, No-FG: 0.7488, Original: 0.9846


{'architecture': 'vit_small_patch16_224.augreg_in1k',
 'data_type': 'original',
 'seed': 7,
 'variant_accuracies': {'original': 0.9845718693751607,
  'no_fg': 0.7487786063255335,
  'mixed_same': 0.9480586268963744,
  'mixed_rand': 0.8621753664181023},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.7487786063255335,
  'bg_gap': 0.0858832604782721,
  'original_accuracy': 0.9845718693751607},
 'final_val_accuracy': 0.9703075291622482,
 'epochs_run': 4,
 'training_time_in_secs': 433.64022755622864,
 'best_val_accuracy': 0.9734888653234358}

In [19]:
run_vit_experiment(arch_name="vit_small_patch16_224.augreg_in1k", augmented=False, seed=8)


vit_small_patch16_224.augreg_in1k_original_8
[vit_small_patch16_224.augreg_in1k_original_8] Epoch 1/10: train_loss=0.3967  train_acc=0.8778 val_loss=0.1026  val_acc=0.9671
[vit_small_patch16_224.augreg_in1k_original_8] Epoch 2/10: train_loss=0.0415  train_acc=0.9882 val_loss=0.1506  val_acc=0.9650
[vit_small_patch16_224.augreg_in1k_original_8] Epoch 3/10: train_loss=0.0201  train_acc=0.9949 val_loss=0.1269  val_acc=0.9735
[vit_small_patch16_224.augreg_in1k_original_8] Epoch 4/10: train_loss=0.0099  train_acc=0.9973 val_loss=0.1302  val_acc=0.9761
[vit_small_patch16_224.augreg_in1k_original_8] Epoch 5/10: train_loss=0.0056  train_acc=0.9983 val_loss=0.1651  val_acc=0.9708
[vit_small_patch16_224.augreg_in1k_original_8] Epoch 6/10: train_loss=0.0029  train_acc=0.9992 val_loss=0.1608  val_acc=0.9767
[vit_small_patch16_224.augreg_in1k_original_8] Epoch 7/10: train_loss=0.0014  train_acc=0.9997 val_loss=0.1479  val_acc=0.9777
[vit_small_patch16_224.augreg_in1k_original_8] Epoch 8/10: train_

{'architecture': 'vit_small_patch16_224.augreg_in1k',
 'data_type': 'original',
 'seed': 8,
 'variant_accuracies': {'original': 0.9889431730521985,
  'no_fg': 0.7274363589611725,
  'mixed_same': 0.9555155566983801,
  'mixed_rand': 0.8729750578554898},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.7274363589611725,
  'bg_gap': 0.08254049884289028,
  'original_accuracy': 0.9889431730521985},
 'final_val_accuracy': 0.9803817603393425,
 'epochs_run': 10,
 'training_time_in_secs': 1043.6567029953003,
 'best_val_accuracy': 0.9803817603393425}

# ViT Small (Augmentation)

In [20]:
run_vit_experiment(arch_name= "vit_small_patch16_224.augreg_in1k",augmented=True, seed=0)


vit_small_patch16_224.augreg_in1k_augmented_0
[vit_small_patch16_224.augreg_in1k_augmented_0] Epoch 1/10: train_loss=0.7061  train_acc=0.7719 val_loss=0.2051  val_acc=0.9358
[vit_small_patch16_224.augreg_in1k_augmented_0] Epoch 2/10: train_loss=0.2022  train_acc=0.9387 val_loss=0.2058  val_acc=0.9332
[vit_small_patch16_224.augreg_in1k_augmented_0] Epoch 3/10: train_loss=0.1348  train_acc=0.9595 val_loss=0.2577  val_acc=0.9236
[vit_small_patch16_224.augreg_in1k_augmented_0] Epoch 4/10: train_loss=0.0929  train_acc=0.9724 val_loss=0.2776  val_acc=0.9226
[vit_small_patch16_224.augreg_in1k_augmented_0] Early stopping at epoch 4
Test variants: {'original': 0.978657752635639, 'no_fg': 0.6238107482643354, 'mixed_same': 0.935201851375675, 'mixed_rand': 0.9308305476986372}
BG-Gap: 0.0044, No-FG: 0.6238, Original: 0.9787


{'architecture': 'vit_small_patch16_224.augreg_in1k',
 'data_type': 'augmented',
 'seed': 0,
 'variant_accuracies': {'original': 0.978657752635639,
  'no_fg': 0.6238107482643354,
  'mixed_same': 0.935201851375675,
  'mixed_rand': 0.9308305476986372},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.6238107482643354,
  'bg_gap': 0.004371303677037863,
  'original_accuracy': 0.978657752635639},
 'final_val_accuracy': 0.9225874867444327,
 'epochs_run': 4,
 'training_time_in_secs': 2373.2488050460815,
 'best_val_accuracy': 0.9358430540827147}

In [21]:
run_vit_experiment(arch_name= "vit_small_patch16_224.augreg_in1k",augmented=True, seed=339)


vit_small_patch16_224.augreg_in1k_augmented_339
[vit_small_patch16_224.augreg_in1k_augmented_339] Epoch 1/10: train_loss=0.7165  train_acc=0.7672 val_loss=0.1982  val_acc=0.9380
[vit_small_patch16_224.augreg_in1k_augmented_339] Epoch 2/10: train_loss=0.2042  train_acc=0.9378 val_loss=0.2575  val_acc=0.9221
[vit_small_patch16_224.augreg_in1k_augmented_339] Epoch 3/10: train_loss=0.1336  train_acc=0.9597 val_loss=0.2361  val_acc=0.9369
[vit_small_patch16_224.augreg_in1k_augmented_339] Epoch 4/10: train_loss=0.0952  train_acc=0.9723 val_loss=0.3088  val_acc=0.9109
[vit_small_patch16_224.augreg_in1k_augmented_339] Early stopping at epoch 4
Test variants: {'original': 0.9799434301877089, 'no_fg': 0.6078683466186681, 'mixed_same': 0.9416302391360247, 'mixed_rand': 0.9416302391360247}
BG-Gap: 0.0000, No-FG: 0.6079, Original: 0.9799


{'architecture': 'vit_small_patch16_224.augreg_in1k',
 'data_type': 'augmented',
 'seed': 339,
 'variant_accuracies': {'original': 0.9799434301877089,
  'no_fg': 0.6078683466186681,
  'mixed_same': 0.9416302391360247,
  'mixed_rand': 0.9416302391360247},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.6078683466186681,
  'bg_gap': 0.0,
  'original_accuracy': 0.9799434301877089},
 'final_val_accuracy': 0.9109225874867445,
 'epochs_run': 4,
 'training_time_in_secs': 2384.577238559723,
 'best_val_accuracy': 0.9379639448568399}

In [22]:
run_vit_experiment(arch_name= "vit_small_patch16_224.augreg_in1k",augmented=True, seed=1)


vit_small_patch16_224.augreg_in1k_augmented_1
[vit_small_patch16_224.augreg_in1k_augmented_1] Epoch 1/10: train_loss=0.6836  train_acc=0.7796 val_loss=0.2388  val_acc=0.9279
[vit_small_patch16_224.augreg_in1k_augmented_1] Epoch 2/10: train_loss=0.2000  train_acc=0.9385 val_loss=0.2587  val_acc=0.9210
[vit_small_patch16_224.augreg_in1k_augmented_1] Epoch 3/10: train_loss=0.1303  train_acc=0.9605 val_loss=0.2931  val_acc=0.9130
[vit_small_patch16_224.augreg_in1k_augmented_1] Epoch 4/10: train_loss=0.0935  train_acc=0.9725 val_loss=0.3236  val_acc=0.9205
[vit_small_patch16_224.augreg_in1k_augmented_1] Early stopping at epoch 4
Test variants: {'original': 0.978400617125225, 'no_fg': 0.6106968372332219, 'mixed_same': 0.9454872717922345, 'mixed_rand': 0.9426587811776806}
BG-Gap: 0.0028, No-FG: 0.6107, Original: 0.9784


{'architecture': 'vit_small_patch16_224.augreg_in1k',
 'data_type': 'augmented',
 'seed': 1,
 'variant_accuracies': {'original': 0.978400617125225,
  'no_fg': 0.6106968372332219,
  'mixed_same': 0.9454872717922345,
  'mixed_rand': 0.9426587811776806},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.6106968372332219,
  'bg_gap': 0.0028284906145538136,
  'original_accuracy': 0.978400617125225},
 'final_val_accuracy': 0.9204665959703076,
 'epochs_run': 4,
 'training_time_in_secs': 2390.304239988327,
 'best_val_accuracy': 0.9278897136797455}

In [23]:
run_vit_experiment(arch_name= "vit_small_patch16_224.augreg_in1k",augmented=True, seed=2)


vit_small_patch16_224.augreg_in1k_augmented_2
[vit_small_patch16_224.augreg_in1k_augmented_2] Epoch 1/10: train_loss=0.7079  train_acc=0.7701 val_loss=0.2030  val_acc=0.9364
[vit_small_patch16_224.augreg_in1k_augmented_2] Epoch 2/10: train_loss=0.2044  train_acc=0.9368 val_loss=0.2192  val_acc=0.9358
[vit_small_patch16_224.augreg_in1k_augmented_2] Epoch 3/10: train_loss=0.1328  train_acc=0.9600 val_loss=0.2387  val_acc=0.9300
[vit_small_patch16_224.augreg_in1k_augmented_2] Epoch 4/10: train_loss=0.0901  train_acc=0.9734 val_loss=0.2710  val_acc=0.9353
[vit_small_patch16_224.augreg_in1k_augmented_2] Early stopping at epoch 4
Test variants: {'original': 0.978143481614811, 'no_fg': 0.6212393931601954, 'mixed_same': 0.9364875289277449, 'mixed_rand': 0.935973257906917}
BG-Gap: 0.0005, No-FG: 0.6212, Original: 0.9781


{'architecture': 'vit_small_patch16_224.augreg_in1k',
 'data_type': 'augmented',
 'seed': 2,
 'variant_accuracies': {'original': 0.978143481614811,
  'no_fg': 0.6212393931601954,
  'mixed_same': 0.9364875289277449,
  'mixed_rand': 0.935973257906917},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.6212393931601954,
  'bg_gap': 0.0005142710208279055,
  'original_accuracy': 0.978143481614811},
 'final_val_accuracy': 0.9353128313891834,
 'epochs_run': 4,
 'training_time_in_secs': 2389.8983397483826,
 'best_val_accuracy': 0.936373276776246}

In [24]:
run_vit_experiment(arch_name= "vit_small_patch16_224.augreg_in1k",augmented=True, seed=3)


vit_small_patch16_224.augreg_in1k_augmented_3
[vit_small_patch16_224.augreg_in1k_augmented_3] Epoch 1/10: train_loss=0.7082  train_acc=0.7694 val_loss=0.2275  val_acc=0.9327
[vit_small_patch16_224.augreg_in1k_augmented_3] Epoch 2/10: train_loss=0.2051  train_acc=0.9377 val_loss=0.1932  val_acc=0.9411
[vit_small_patch16_224.augreg_in1k_augmented_3] Epoch 3/10: train_loss=0.1342  train_acc=0.9595 val_loss=0.2433  val_acc=0.9321
[vit_small_patch16_224.augreg_in1k_augmented_3] Epoch 4/10: train_loss=0.0924  train_acc=0.9725 val_loss=0.2668  val_acc=0.9274
[vit_small_patch16_224.augreg_in1k_augmented_3] Epoch 5/10: train_loss=0.0661  train_acc=0.9816 val_loss=0.3810  val_acc=0.9125
[vit_small_patch16_224.augreg_in1k_augmented_3] Early stopping at epoch 5
Test variants: {'original': 0.9742864489586012, 'no_fg': 0.5042427359218308, 'mixed_same': 0.9328876317819491, 'mixed_rand': 0.9298020056569812}
BG-Gap: 0.0031, No-FG: 0.5042, Original: 0.9743


{'architecture': 'vit_small_patch16_224.augreg_in1k',
 'data_type': 'augmented',
 'seed': 3,
 'variant_accuracies': {'original': 0.9742864489586012,
  'no_fg': 0.5042427359218308,
  'mixed_same': 0.9328876317819491,
  'mixed_rand': 0.9298020056569812},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.5042427359218308,
  'bg_gap': 0.0030856261249678774,
  'original_accuracy': 0.9742864489586012},
 'final_val_accuracy': 0.9125132555673383,
 'epochs_run': 5,
 'training_time_in_secs': 2991.0080053806305,
 'best_val_accuracy': 0.9411452810180275}

In [25]:
run_vit_experiment(arch_name= "vit_small_patch16_224.augreg_in1k",augmented=True, seed=4)


vit_small_patch16_224.augreg_in1k_augmented_4
[vit_small_patch16_224.augreg_in1k_augmented_4] Epoch 1/10: train_loss=0.7513  train_acc=0.7534 val_loss=0.2056  val_acc=0.9358
[vit_small_patch16_224.augreg_in1k_augmented_4] Epoch 2/10: train_loss=0.2042  train_acc=0.9380 val_loss=0.2089  val_acc=0.9332
[vit_small_patch16_224.augreg_in1k_augmented_4] Epoch 3/10: train_loss=0.1333  train_acc=0.9595 val_loss=0.2415  val_acc=0.9311
[vit_small_patch16_224.augreg_in1k_augmented_4] Epoch 4/10: train_loss=0.0920  train_acc=0.9735 val_loss=0.2818  val_acc=0.9290
[vit_small_patch16_224.augreg_in1k_augmented_4] Early stopping at epoch 4
Test variants: {'original': 0.9755721265106712, 'no_fg': 0.56672666495243, 'mixed_same': 0.9385446130110568, 'mixed_rand': 0.9310876832090512}
BG-Gap: 0.0075, No-FG: 0.5667, Original: 0.9756


{'architecture': 'vit_small_patch16_224.augreg_in1k',
 'data_type': 'augmented',
 'seed': 4,
 'variant_accuracies': {'original': 0.9755721265106712,
  'no_fg': 0.56672666495243,
  'mixed_same': 0.9385446130110568,
  'mixed_rand': 0.9310876832090512},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.56672666495243,
  'bg_gap': 0.00745692980200563,
  'original_accuracy': 0.9755721265106712},
 'final_val_accuracy': 0.9289501590668081,
 'epochs_run': 4,
 'training_time_in_secs': 2394.873898744583,
 'best_val_accuracy': 0.9358430540827147}

In [26]:
run_vit_experiment(arch_name= "vit_small_patch16_224.augreg_in1k",augmented=True, seed=5)


vit_small_patch16_224.augreg_in1k_augmented_5
[vit_small_patch16_224.augreg_in1k_augmented_5] Epoch 1/10: train_loss=0.7213  train_acc=0.7674 val_loss=0.2013  val_acc=0.9411
[vit_small_patch16_224.augreg_in1k_augmented_5] Epoch 2/10: train_loss=0.2011  train_acc=0.9382 val_loss=0.1993  val_acc=0.9401
[vit_small_patch16_224.augreg_in1k_augmented_5] Epoch 3/10: train_loss=0.1328  train_acc=0.9600 val_loss=0.2481  val_acc=0.9274
[vit_small_patch16_224.augreg_in1k_augmented_5] Epoch 4/10: train_loss=0.0918  train_acc=0.9731 val_loss=0.3407  val_acc=0.9104
[vit_small_patch16_224.augreg_in1k_augmented_5] Early stopping at epoch 4
Test variants: {'original': 0.976857804062741, 'no_fg': 0.6454101311391103, 'mixed_same': 0.9493443044484443, 'mixed_rand': 0.9434301877089226}
BG-Gap: 0.0059, No-FG: 0.6454, Original: 0.9769


{'architecture': 'vit_small_patch16_224.augreg_in1k',
 'data_type': 'augmented',
 'seed': 5,
 'variant_accuracies': {'original': 0.976857804062741,
  'no_fg': 0.6454101311391103,
  'mixed_same': 0.9493443044484443,
  'mixed_rand': 0.9434301877089226},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.6454101311391103,
  'bg_gap': 0.005914116739521691,
  'original_accuracy': 0.976857804062741},
 'final_val_accuracy': 0.9103923647932132,
 'epochs_run': 4,
 'training_time_in_secs': 2393.8455443382263,
 'best_val_accuracy': 0.9411452810180275}

In [27]:
run_vit_experiment(arch_name= "vit_small_patch16_224.augreg_in1k",augmented=True, seed=6)


vit_small_patch16_224.augreg_in1k_augmented_6
[vit_small_patch16_224.augreg_in1k_augmented_6] Epoch 1/10: train_loss=0.7097  train_acc=0.7696 val_loss=0.2178  val_acc=0.9358
[vit_small_patch16_224.augreg_in1k_augmented_6] Epoch 2/10: train_loss=0.2054  train_acc=0.9359 val_loss=0.2369  val_acc=0.9348
[vit_small_patch16_224.augreg_in1k_augmented_6] Epoch 3/10: train_loss=0.1306  train_acc=0.9607 val_loss=0.2708  val_acc=0.9263
[vit_small_patch16_224.augreg_in1k_augmented_6] Epoch 4/10: train_loss=0.0933  train_acc=0.9724 val_loss=0.2544  val_acc=0.9305
[vit_small_patch16_224.augreg_in1k_augmented_6] Early stopping at epoch 4
Test variants: {'original': 0.9758292620210851, 'no_fg': 0.565183851889946, 'mixed_same': 0.9305734121882232, 'mixed_rand': 0.9298020056569812}
BG-Gap: 0.0008, No-FG: 0.5652, Original: 0.9758


{'architecture': 'vit_small_patch16_224.augreg_in1k',
 'data_type': 'augmented',
 'seed': 6,
 'variant_accuracies': {'original': 0.9758292620210851,
  'no_fg': 0.565183851889946,
  'mixed_same': 0.9305734121882232,
  'mixed_rand': 0.9298020056569812},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.565183851889946,
  'bg_gap': 0.0007714065312419693,
  'original_accuracy': 0.9758292620210851},
 'final_val_accuracy': 0.9305408271474019,
 'epochs_run': 4,
 'training_time_in_secs': 2391.1567890644073,
 'best_val_accuracy': 0.9358430540827147}

In [28]:
run_vit_experiment(arch_name= "vit_small_patch16_224.augreg_in1k",augmented=True, seed=7)


vit_small_patch16_224.augreg_in1k_augmented_7
[vit_small_patch16_224.augreg_in1k_augmented_7] Epoch 1/10: train_loss=0.6994  train_acc=0.7757 val_loss=0.2037  val_acc=0.9459
[vit_small_patch16_224.augreg_in1k_augmented_7] Epoch 2/10: train_loss=0.2063  train_acc=0.9370 val_loss=0.2196  val_acc=0.9295
[vit_small_patch16_224.augreg_in1k_augmented_7] Epoch 3/10: train_loss=0.1316  train_acc=0.9596 val_loss=0.2846  val_acc=0.9183
[vit_small_patch16_224.augreg_in1k_augmented_7] Epoch 4/10: train_loss=0.0895  train_acc=0.9733 val_loss=0.2842  val_acc=0.9173
[vit_small_patch16_224.augreg_in1k_augmented_7] Early stopping at epoch 4
Test variants: {'original': 0.978914888146053, 'no_fg': 0.653638467472358, 'mixed_same': 0.9449730007714066, 'mixed_rand': 0.9377732064798149}
BG-Gap: 0.0072, No-FG: 0.6536, Original: 0.9789


{'architecture': 'vit_small_patch16_224.augreg_in1k',
 'data_type': 'augmented',
 'seed': 7,
 'variant_accuracies': {'original': 0.978914888146053,
  'no_fg': 0.653638467472358,
  'mixed_same': 0.9449730007714066,
  'mixed_rand': 0.9377732064798149},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.653638467472358,
  'bg_gap': 0.007199794291591677,
  'original_accuracy': 0.978914888146053},
 'final_val_accuracy': 0.9172852598091198,
 'epochs_run': 4,
 'training_time_in_secs': 2402.3557465076447,
 'best_val_accuracy': 0.9459172852598091}

In [29]:
run_vit_experiment(arch_name= "vit_small_patch16_224.augreg_in1k",augmented=True, seed=8)


vit_small_patch16_224.augreg_in1k_augmented_8
[vit_small_patch16_224.augreg_in1k_augmented_8] Epoch 1/10: train_loss=0.6634  train_acc=0.7893 val_loss=0.2058  val_acc=0.9396
[vit_small_patch16_224.augreg_in1k_augmented_8] Epoch 2/10: train_loss=0.2031  train_acc=0.9372 val_loss=0.2583  val_acc=0.9199
[vit_small_patch16_224.augreg_in1k_augmented_8] Epoch 3/10: train_loss=0.1314  train_acc=0.9601 val_loss=0.2495  val_acc=0.9215
[vit_small_patch16_224.augreg_in1k_augmented_8] Epoch 4/10: train_loss=0.0919  train_acc=0.9727 val_loss=0.3208  val_acc=0.9183
[vit_small_patch16_224.augreg_in1k_augmented_8] Early stopping at epoch 4
Test variants: {'original': 0.9766006685523271, 'no_fg': 0.6765235278992029, 'mixed_same': 0.9382874775006428, 'mixed_rand': 0.9357161223965029}
BG-Gap: 0.0026, No-FG: 0.6765, Original: 0.9766


{'architecture': 'vit_small_patch16_224.augreg_in1k',
 'data_type': 'augmented',
 'seed': 8,
 'variant_accuracies': {'original': 0.9766006685523271,
  'no_fg': 0.6765235278992029,
  'mixed_same': 0.9382874775006428,
  'mixed_rand': 0.9357161223965029},
 'reliance_metrics': {'chance_accuracy': 0.1111111111111111,
  'no_fg_accuracy': 0.6765235278992029,
  'bg_gap': 0.002571355104139861,
  'original_accuracy': 0.9766006685523271},
 'final_val_accuracy': 0.9183457051961824,
 'epochs_run': 4,
 'training_time_in_secs': 2391.6704239845276,
 'best_val_accuracy': 0.9395546129374337}

# Short Summary

In [30]:
results_path = RESULTS / "vit_small_results.json"
with open(results_path, "r") as f:
    results = json.load(f)

rows = []
for r in results:
    row = {"architecture": r["architecture"], "data_type": r["data_type"], "seed": r["seed"], "best_val_accuracy": r["best_val_accuracy"], "epochs_run": r["epochs_run"]}
    row.update(r["reliance_metrics"])
    row["original_accuracy"] = r["variant_accuracies"].get("original")
    rows.append(row)

df = pd.DataFrame(rows)
summary = df.groupby(["architecture", "data_type"]).agg(
    mean_bg_gap =("bg_gap", "mean"), std_bg_gap =("bg_gap", "std"),
    mean_no_fg =("no_fg_accuracy", "mean"), std_no_fg =("no_fg_accuracy", "std"),
    mean_original =("original_accuracy", "mean"), std_original =("original_accuracy", "std"),
)
summary

mean_bg_gap  std_bg_gap  \
architecture                      data_type                            
vit_small_patch16_224.augreg_in1k augmented     0.003471    0.002708   
                                  original      0.078503    0.005198   

                                             mean_no_fg  std_no_fg  \
architecture                      data_type                          
vit_small_patch16_224.augreg_in1k augmented    0.607534   0.050454   
                                  original     0.723888   0.017227   

                                             mean_original  std_original  
architecture                      data_type                               
vit_small_patch16_224.augreg_in1k augmented       0.977321      0.001772  
                                  original        0.986706      0.002357

# Show TensorBoard

In [ ]:
# %load_ext tensorboard
# %tensorboard --logdir ./runs_10_epochs


# Attention Rollout

In [6]:
from cory_attention_rollout_test import get_vit_attention_rollout
from PIL import Image
import pathlib
import torch

model = ViTModel("vit_small_patch16_224.augreg_in1k", num_classes=9, pretrained=False)
# Print out the parameter count
print(f"ViT-S/16 Parameter Count: {sum(p.numel() for p in model.parameters()):,}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS = pathlib.Path("results_10_epochs")

mixed_rand_test = fix_test_labels(ImageFolderEval(TEST_INSTANCES / "mixed_rand" / "val", image_size=IMAGE_SIZE, mean = VIT_MEAN, std=VIT_STD), train_class_to_idx)

def perform_attn_rollout(augmented=False, img_index=339):
    """
    Attention rollout for one Mixed-Rand test image using the seed-0 weights. Prints the true and predicted label
    and saves the overlay to figures_10_epochs/
    @param augmented: True for the augmented model, False for the original model
    @param img_index: index of the image in the Mixed-Rand test set
    """
    if augmented:
        model.load_state_dict(torch.load(RESULTS / "vit_small_patch16_224.augreg_in1k_augmented_0_weights.pt", map_location=device))
    else:
        model.load_state_dict(torch.load(RESULTS / "vit_small_patch16_224.augreg_in1k_original_0_weights.pt", map_location=device))
    model.to(device)
    image, label = mixed_rand_test[img_index]

    overlay, predicted = get_vit_attention_rollout(model, image, device = device)
    print(f"True Label: {label}, Predicted: {predicted}")
    Image.fromarray(overlay).save(FIGURES / f"attn_ViT_small_rollout_{'augmented' if augmented else 'original'}_{img_index}.png")

perform_attn_rollout(augmented=False, img_index=1023)
perform_attn_rollout(augmented= True, img_index=1023)

C:\Users\minio\PycharmProjects\JupyterProject2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ViT-S/16 Parameter Count: 21,669,129
True Label: 8, Predicted: 4
True Label: 8, Predicted: 8


# Inference Time Counter

In [32]:
import torch.utils.benchmark as benchmark
# https://docs.pytorch.org/docs/2.14/benchmark_utils.html
# https://docs.pytorch.org/tutorials/recipes/recipes/benchmark.html

def benchmark_model(model,name):
    """
    Median time of one forward pass on a random batch of BATCH_SIZE images, and the peak GPU memory used.
    The per-image inference times in the report come from helper_code.ipynb, cell 8.
    @param model: the model to time
    @param name: name printed with the result
    """
    model=model.to(device).eval()
    x = torch.randn(BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE, device = device)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    timer = benchmark.Timer(stmt="with torch.no_grad(): model(x)", globals={"model": model, "x": x, "torch": torch})
    result = timer.blocked_autorange()
    if device.type == "cuda":
        print(f"{name}: took {result.median*1000} ms, with a peak memory usage of: {torch.cuda.max_memory_allocated() / 1e6} MegaBytes")
    else:
        print(f"{name}: took {result.median*1000} ms")

benchmark_model(ViTModel("vit_small_patch16_224.augreg_in1k", num_classes=9, pretrained=True), "vit_small_patch16_224.augreg_in1k")

vit_small_patch16_224.augreg_in1k: took 25.524800003040582 ms, with a peak memory usage of: 327.10144 MegaBytes
